# 🚀 MorphoSpeech-LLM: Türkçe Otomatik Konuşma Tanıma (ASR) SOTA Eğitimi

Bu Google Colab notebook'u, **Türkçe Otomatik Konuşma Tanıma (ASR)** alanında akademik SOTA (State of the Art) sonuçları aşmak üzere tasarlanmıştır.

### 🎯 Hedefler:
1. Google/Meta **FLEURS Türkçe** ve Mozilla **Common Voice Türkçe** verisetlerinde SOTA (WER %6.8) altını hedeflemek.
2. **LoRA (PEFT)** ile **Whisper-Large-v3** veya **Whisper-Small** modelini Türkçe ek birleşimlerine (agglutinative morphology) uyarlamak.
3. Eğitim sonunda otomatik WER (%) ve CER (%) metriklerini hesaplayıp en iyi checkpoint'i kaydetmek.

> ⚠️ **Not:** Menüden **Çalışma Zamanı -> Çalışma zamanı türünü değiştir** seçeneğine girip **T4 GPU**, **L4 GPU** veya **A100 GPU** seçtiğinizden emin olun.

## 📦 Adım 1: Bağımlılıkların Yüklenmesi ve HF_TOKEN Ayarı

In [1]:
# GPU Durumunu Kontrol Etme
!nvidia-smi

# Gerekli kütüphaneleri yükleme
!pip install -q torch torchaudio transformers 'datasets<3.0.0' evaluate jiwer librosa soundfile accelerate peft optuna morfessor sentencepiece

# Optional HF_TOKEN loading from Colab Secrets
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('🔑 HF_TOKEN başarıyla aktifleştirildi!')
except Exception as e:
    print('ℹ️ HF_TOKEN olmadan anonim olarak devam ediliyor (Sorunsuz çalışır).')

Fri Oct  2 11:19:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 🛠️ Adım 2: Reponun Klonlanması ve PYTHONPATH Ayarı

In [2]:
import os, sys
if not os.path.exists('turkish-asr'):
    !git clone https://github.com/gokhanersoy/turkish-asr.git
    %cd turkish-asr
else:
    %cd turkish-asr
    !git pull

# Set PYTHONPATH to project root
sys.path.insert(0, os.getcwd())
os.environ['PYTHONPATH'] = os.getcwd()
print('✅ Proje dizini ve PYTHONPATH başarıyla ayarlandı:', os.getcwd())

Cloning into 'turkish-asr'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 90 (delta 27), reused 88 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 66.30 KiB | 2.46 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/turkish-asr
✅ Proje dizini ve PYTHONPATH başarıyla ayarlandı: /content/turkish-asr


In [3]:
#!pip install -U torchao peft

In [4]:
#%cd /content
#!rm -rf turkish-asr
#!git clone https://github.com/gokhanersoy/turkish-asr.git
#%cd turkish-asr

## 🏋️‍♂️ Adım 3: Türkçe SOTA Fine-Tuning Eğitiminin Başlatılması

Colab GPU (A100 / L4 / T4) üzerinde **Whisper-Small** (veya `openai/whisper-large-v3`) modelini FLEURS Türkçe veriseti üzerinde eğitiyoruz.

In [5]:
# Fine-Tuning Eğitimi (3 Epoch, LoRA PEFT Hizalandırma)
!PYTHONPATH=. python scripts/train_morphospeech.py \
    --model openai/whisper-small \
    --dataset fleurs \
    --train-samples 300 \
    --val-samples 50 \
    --epochs 3 \
    --batch-size 8 \
    --learning-rate 1e-4 \
    --output-dir checkpoints/morphospeech_sota

INFO:__main__:==================================================
INFO:__main__:  TÜRKÇE ASR SOTA EĞİTİMİ (MORPHOSPEECH FINE-TUNING) 
INFO:__main__:==================================================
INFO:__main__:Model        : openai/whisper-small
INFO:__main__:Veriseti     : fleurs
INFO:__main__:Eğitim Sayısı: 300 ses
INFO:__main__:Doğrulama    : 50 ses
INFO:__main__:Epoch Sayısı : 3
INFO:src.datasets.loader:Loading FLEURS Turkish dataset (config='tr_tr', split=train, streaming=True)...
INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/fleurs "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/fleurs "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/fleurs/revision/70bb2e84b976b7e960aa89f1c648e09c59f894dd "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/fleu

## 📊 Adım 4: Eğitilen SOTA Modelinin Karşılaştırmalı Değerlendirilmesi (WER & CER)

In [6]:
# Eğitilen en iyi checkpoint üzerinde benchmark testi koşturma
!PYTHONPATH=. python scripts/run_benchmark.py \
    --model checkpoints/morphospeech_sota/best_sota_model \
    --dataset fleurs \
    --split test \
    --max-samples 50

INFO:__main__:==================================================
INFO:__main__:         TÜRKÇE ASR BENCHMARK RUNNER               
INFO:__main__:==================================================
INFO:__main__:Model: checkpoints/morphospeech_sota/best_sota_model
INFO:__main__:Dataset: fleurs (split: test, max_samples: 50)
INFO:src.datasets.loader:Loading FLEURS Turkish dataset (config='tr_tr', split=test, streaming=True)...
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/fleurs "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/fleurs "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/fleurs/revision/70bb2e84b976b7e960aa89f1c648e09c59f894dd "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/google/fleurs/tree/70bb2e84b976b7e960aa89f1c648e09c59f894dd?recursive=false&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/dat